# 01 · Data and validation

**Question:** What would a convincing test of rule generalization look like?

The official training set contains only two rules. A random split alone mostly tests familiar policies. We therefore report seen-rule and held-out-rule performance separately.

In [ ]:
import os
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML, FileLink
from jigsaw_rules.data import load_data, audit
from jigsaw_rules.runtime import Progress, environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = ["#087f8c", "#bd633b", "#334ea0", "#70923b"]
display(HTML("<div style='padding:18px;background:#edf6f5;border-left:5px solid #087f8c'>"
             "<b>Jigsaw research workspace</b><br>Every result must identify its data and validation protocol.</div>"))
print("Project:", root)


In [ ]:
train, test, sample = load_data(root / 'data/raw')
overview = audit(train, test)
display(pd.DataFrame(overview['by_rule']))
print('Duplicate bodies:', overview['duplicate_training_bodies'])
print('Train/preview-test overlap:', overview['train_test_body_overlap'])

## Class balance and comment length
Rule imbalance and long comments affect both measurement and future transformer token budgets. These plots use only supplied data.

In [ ]:
by_rule = pd.DataFrame(overview['by_rule'])
fig = px.bar(by_rule, x='rule', y='mean', color='rule', title='Violation prevalence by rule', labels={'mean': 'Violation rate', 'rule': 'Rule'})
fig.update_layout(showlegend=False, height=420)
fig.update_yaxes(range=[0, 1])
fig.show()
lengths = train.assign(characters=train.body.str.len())
px.histogram(lengths, x='characters', color='rule', nbins=35, title='Comment length distribution', barmode='overlay', opacity=.6).show()

## Freeze the validation design
1. **Seen rule:** stratify by rule and label; group normalized duplicate comments.
2. **Held-out rule:** train on other rules and evaluate the omitted rule.
3. Purge training rows if their body or support examples contain a validation body.
4. Fit vocabulary only on the remaining training rows.

Provided validation examples are legitimate model inputs. We do not convert validation examples into new supervised training rows. Exact-text checks do not detect every paraphrase or shared source. Two rules are too few to estimate transfer across all community policies.

In [ ]:
from jigsaw_rules.splits import make_splits
rows = []
for protocol in ['seen_rule', 'heldout_rule']:
    for fold, (ti, vi, purged) in enumerate(make_splits(train, protocol, folds=3, seed=2025)):
        rows.append({'Protocol': protocol, 'Fold': fold, 'Training rows': len(ti), 'Validation rows': len(vi), 'Purged rows': purged})
display(pd.DataFrame(rows))

## Metric contract
The official overview calls the metric **column-averaged AUC**. We report **rule macro AUC** (equal-weight mean of rule-specific ROC AUC), consistent with published descriptions of the challenge, and also **pooled AUC** so the distinction stays visible. The overview does not expose executable scorer code.

Secondary diagnostics: average precision, log loss, Brier score, calibration error, and precision/recall/F1 at a predeclared 0.5 threshold. We never replace an undefined single-class AUC with a favorable number.

Next: **02_baseline_and_review.ipynb**.